Write code for Hamming(7,4) and Hamming(8,4)

In [ ]:
!jupyter nbconvert --to script communityGraphFunctions.ipynb quantumUtility.ipynb



In [ ]:
from communityGraphFunctions import *
from quantumUtility import *

In [ ]:
# Define global parameters for the problem
N = 6  # Number of nodes in the graph
P = 1  # Number of QAOA layers
K = 4 #Number of communities
LAMBDA_PENALTY = 3
IBM_KEY = '' # IBM Quantum API key for authenticating with Qiskit Runtime services
# Initial Parameters for optimization
INITIAL_GAMMA = np.pi # Initial value for the gamma parameter in QAOA, often chosen as pi
INITIAL_BETA = np.pi/2 # Initial value for the beta parameter in QAOA, often chosen as pi/2

# Standard Hamming(7,4) parity check sets (0-indexed bit positions).
# Each bit b has a unique syndrome equal to the binary representation of (b+1),
# so syndrome (s2,s1,s0) directly gives the error position:
#   P1 covers bits where (b+1) has bit-0 set: positions 0,2,4,6
#   P2 covers bits where (b+1) has bit-1 set: positions 1,2,5,6
#   P3 covers bits where (b+1) has bit-2 set: positions 3,4,5,6
# This ensures all 7 non-zero 3-bit syndromes are reachable, including (1,1,1).
PARITY_CODES_74 = [
    [0, 2, 4, 6],  # P1
    [1, 2, 5, 6],  # P2
    [3, 4, 5, 6],  # P3
]

# Extended Hamming(8,4): Hamming(7,4) parity checks plus an overall parity check.
# The overall parity check (P4) covers all 8 bits and allows distinguishing
# single-bit errors (correctable) from double-bit errors (detectable only).
PARITY_CODES_84 = [
    [0, 2, 4, 6],           # P1
    [1, 2, 5, 6],           # P2
    [3, 4, 5, 6],           # P3
    [0, 1, 2, 3, 4, 5, 6, 7],  # P4: overall parity
]

**Modularity Penality for Hamming**

$\mathrm{H_{mod}} = - \sum_{q=1}^{numbits}\sum_{(i,j) \in \mathcal{E}} \left(A_{ij} - \frac{k_i k_j}{2m}\right) Z_{i,q} Z_{j,q} $


In [ ]:
def create_modularity_penalty_hamiltonian(G, N, B, K, no_of_nodes_in_supernode):
    """Creates the Modularity Penalty Hamiltonian for the Hamming(7,4) and Hamming(8,4) encoding.

    Implements:
        H_mod = -sum_{q=0}^{numbits-1} sum_{(i,j) in E} B_ij * Z_{i,q} Z_{j,q}

    Each (edge, bit) pair contributes one ZZ term with coefficient -B_ij.

    Qubit mapping: node i, bit q  ->  i * no_of_nodes_in_supernode + q

    Args:
        G (networkx.Graph): The graph representing the problem.
        N (int): Number of nodes in the graph.
        B (numpy.ndarray): Modularity matrix of shape (N, N), where B[i,j] = A_ij - k_i*k_j/(2m).
        K (int): Number of communities (unused; kept for API consistency).
        no_of_nodes_in_supernode (int): Qubits per node / codeword length (7 or 8).

    Returns:
        tuple:
            - num_qubits (int): Total number of qubits (N * no_of_nodes_in_supernode).
            - mp_hamiltonian (SparsePauliOp): The Modularity Penalty Hamiltonian.
    """
    num_qubits = N * no_of_nodes_in_supernode

    m = G.number_of_edges()
    if m == 0:
        print("Graph has no edges. Modularity Penalty Hamiltonian is trivially zero.")
        return num_qubits, SparsePauliOp.from_list([], num_qubits=num_qubits)

    pauli_terms = []

    for i in range(N):
        for j in range(N):
            B_ij = B[i, j]
            
            if abs(B_ij) < 1e-12:
                continue

            for q in range(no_of_nodes_in_supernode):
                pauli_str = ['I'] * num_qubits
                pauli_str[i * no_of_nodes_in_supernode + q] = 'Z'
                pauli_str[j * no_of_nodes_in_supernode + q] = 'Z'
                pauli_terms.append((''.join(reversed(pauli_str)), -1 * B_ij))

    mp_hamiltonian = SparsePauliOp.from_list(pauli_terms, num_qubits=num_qubits).simplify()

    print(f"\nModularity Penalty Hamiltonian (Hamming {no_of_nodes_in_supernode},4) constructed for {num_qubits} qubits.")
    print(f"Number of terms after simplification: {len(mp_hamiltonian)}")

    return num_qubits, mp_hamiltonian

**Parity Penalty**

$\mathrm{PP} = - \lambda \sum_{i \in \mathcal{V}} \sum_{P \in \mathcal{P}} \left(\prod_{q \in P} Z_{i,q}\right)$

In [ ]:
def create_parity_penalty_hamiltonian(N, no_of_nodes_in_supernode, PARITY_CHECKS, lambda_penalty=1.0):
    """Creates the Parity Penalty Hamiltonian for the Hamming(7,4) encoding.

    Implements:
        PP = lambda * sum_{i in V} sum_{P in P} (1 - prod_{q in P} Z_{i,q})

    Each parity check P contributes two terms per node i:
        - constant term:       +lambda * I
        - multi-qubit Z term:  -lambda * Z_{i,q1} Z_{i,q2} ... Z_{i,q|P|}

    Why lambda_penalty?
    -------------------
    Without scaling, the parity penalty and the modularity term may have
    very different energy scales, causing the optimiser to ignore whichever
    is smaller.  lambda_penalty amplifies the parity constraint so it
    dominates over invalid (non-codeword) states while still leaving the
    modularity term visible.

    A good starting value: lambda_penalty ~ max|B_ij| * 7 * |E| / (3 * N)
    i.e. roughly the total energy scale of the modularity term divided by
    the number of parity checks.  In practice, tune it so that:
        lambda_penalty * 3 (checks) >> max modularity energy per node

    
    Qubit mapping: node i, bit q  ->  i * no_of_nodes_in_supernode + q

    Args:
        N (int): Number of nodes in the graph.
        no_of_nodes_in_supernode (int): Number of nodes in a supernode.
        PARITY_CHECKS (list(list(int))): A list of lists containing parity check syndromes for Hamming(7,4) or Hamming(8,4)
        lambda_penalty (float): Penalty strength multiplier. Default 1.0
            (reproduces original behaviour). Increase to enforce parity
            constraints more strongly relative to the modularity term.

    Returns:
        tuple:
            - num_qubits (int): Total number of qubits (N * no_of_nodes_in_supernode).
            - pp_hamiltonian (SparsePauliOp): The Parity Penalty Hamiltonian.
    """
    num_qubits = N * no_of_nodes_in_supernode

    pauli_terms = []

    for i in range(N):
        for parity_set in PARITY_CHECKS:
            # Multi-qubit Z term: -lambda * Z_{i,q1} Z_{i,q2} ... Z_{i,q|P|}
            pauli_str = ['I'] * num_qubits
            for q in parity_set:
                pauli_str[i * no_of_nodes_in_supernode + q] = 'Z'
            pauli_terms.append((''.join(reversed(pauli_str)), -lambda_penalty))

    pp_hamiltonian = SparsePauliOp.from_list(pauli_terms, num_qubits=num_qubits).simplify()

    print(f"\nParity Penalty Hamiltonian (Hamming {no_of_nodes_in_supernode},4) constructed for {num_qubits} qubits.")
    print(f"lambda_penalty = {lambda_penalty}")
    print(f"Number of terms after simplification: {len(pp_hamiltonian)}")

    return num_qubits, pp_hamiltonian

In [ ]:
def group_nodes_by_community(G, final_answer, no_of_nodes_in_supernode):
    """Decode the binary‑encoded bitstring produced by the QAOA run and
    colour the nodes of `G` according to the community they belong to.

    Parameters
    ----------
    G : networkx.Graph
        The original graph whose nodes are numbered `0 … N‑1`.
    final_answer : list of str
        A list of `'0'`/`'1'` characters of length `N*num_bits` returned by
        `sample_output`.  The bits for node `i` occupy
        `final_answer[i*num_bits:(i+1)*num_bits]`.
    no_of_nodes_in_supernode : int
        The number of communities that was requested; used to clamp the
        decoded integer to a valid range.

    The function sets a node attribute `"community"` on `G` and then calls
    `display_graph` to show the coloured graph.
    """
    N = G.number_of_nodes()
    if N == 0:
        return

    num_bits = len(final_answer) // N
    assignment = {}
    for i in range(N):
        bits = final_answer[i * num_bits : (i + 1) * num_bits]
        value = int("".join(bits), 2)
        if value >= no_of_nodes_in_supernode:
            # wrap/clip the value into the requested community range
            value = value % no_of_nodes_in_supernode
        assignment[i] = value

    nx.set_node_attributes(G, assignment, name="community")

    # generate a colour list (tab10 has at least 10 distinct colours)
    cmap = plt.get_cmap("tab10")
    colors = [cmap(assignment[i] % cmap.N) for i in range(N)]

    display_graph(
        G,
        title="Detected communities (binary encoding)",
        node_colors=colors
    )

In [ ]:
def get_data_bit_indices(no_of_nodes_in_supernode):
    """Return 0-indexed positions of the 4 data bits in a Hamming(n,4) codeword.

    Parity bits sit at positions where (index+1) is a power of 2, i.e.
    (index+1) & index == 0.  All other positions are data bits.

        Hamming(7,4): parity @ {0,1,3}   → data @ [2, 4, 5, 6]
        Hamming(8,4): parity @ {0,1,3,7} → data @ [2, 4, 5, 6]

    Args:
        no_of_nodes_in_supernode (int): Codeword length (7 or 8).

    Returns:
        list of int: Sorted data-bit positions.
    """
    return [i for i in range(no_of_nodes_in_supernode) if (i + 1) & i != 0]


def sample_output(G, optimized_circuit, backend, num_qubits, no_of_nodes_in_supernode,
                  parity_checks, shots=10000):
    """Sample the optimised QAOA circuit, syndrome-correct each codeword,
    and assign communities from the 4 data bits only (parity bits excluded).

    Pipeline
    --------
    1. Transpile and sample the circuit (Qiskit Sampler).
    2. Take the most probable bitstring.
    3. Apply Hamming syndrome correction via correct_hamming_errors().
    4. For each node, read data bits at positions given by get_data_bit_indices().
    5. Interpret those 4 data bits as a binary number → community = value % K.
    6. Colour and display the graph.

    Bit ordering (Qiskit little-endian)
    ------------------------------------
    get_int_counts() returns integers where qubit 0 is the LSB.
    np.binary_repr gives MSB-first strings, so we reverse to get
    index j ↔ qubit j.

    Args:
        G (networkx.Graph): The problem graph.
        optimized_circuit (QuantumCircuit): Circuit with optimal parameters bound.
        backend: AerSimulator or IBM backend.
        num_qubits (int): Total qubits = N * no_of_nodes_in_supernode.
        no_of_nodes_in_supernode (int): Codeword length per node (7 or 8).
        parity_checks (list of list of int): Parity check index sets for syndrome decoding.
        shots (int): Number of measurement shots. Default 10 000.

    Returns:
        list of str: Syndrome-corrected bitstring ('0'/'1'), length num_qubits.
    """
    n_nodes = G.number_of_nodes()
    bits_per_node = num_qubits // n_nodes
    data_indices = get_data_bit_indices(no_of_nodes_in_supernode)

    # ── 1. Re-transpile ───────────────────────────────────────────────────────
    final_circuit = transpile(optimized_circuit, backend)

    # ── 2. Sample ─────────────────────────────────────────────────────────────
    sampler = Sampler(mode=backend)
    sampler.options.default_shots = shots
    job = sampler.run([final_circuit], shots=shots)
    counts_int = job.result()[0].data.meas.get_int_counts()

    total = sum(counts_int.values())
    dist_int = {k: v / total for k, v in counts_int.items()}

    # ── 3. Most probable outcome ───────────────────────────────────────────────
    most_likely_int = max(dist_int, key=dist_int.get)
    print(f"Most probable outcome : integer {most_likely_int}  "
          f"(probability {dist_int[most_likely_int]:.4f})")

    bitstring_msb = np.binary_repr(most_likely_int, width=num_qubits)
    bitstring = list(reversed(bitstring_msb))   # index j = qubit j
    final_answer = [str(b) for b in bitstring]

    print(f"Raw bitstring (qubit-0 first): {''.join(final_answer)}")

    # ── 4. Syndrome correction ────────────────────────────────────────────────
    final_answer = correct_hamming_errors(final_answer, n_nodes, no_of_nodes_in_supernode, parity_checks)

    # ── 5. Decode communities from data bits only ─────────────────────────────
    print(f"\nCommunity assignment from data bits {data_indices} (K={K}):")
    print(f"{'Node':<6} {'Codeword':<{bits_per_node + 2}} {'Data bits':<10} {'Value':<7} Community")
    print("-" * (bits_per_node + 32))

    assignment = {}
    for i in range(n_nodes):
        codeword   = final_answer[i * bits_per_node : (i + 1) * bits_per_node]
        data_bits  = [codeword[q] for q in data_indices]
        value      = int(''.join(data_bits), 2)
        community  = value % K
        assignment[i] = community
        print(f"{i:<6} {''.join(codeword):<{bits_per_node + 2}} {''.join(data_bits):<10} {value:<7} {community}")

    # ── 6. Colour the graph ───────────────────────────────────────────────────
    nx.set_node_attributes(G, assignment, name="community")
    cmap   = plt.get_cmap("tab10")
    colors = [cmap(assignment[i] % cmap.N) for i in range(n_nodes)]
    display_graph(
        G,
        title=f"Detected communities — Hamming({no_of_nodes_in_supernode},4), data bits only",
        node_colors=colors
    )

    return final_answer

In [ ]:
def build_syndrome_table(parity_checks, codeword_length):
    """Map every single-bit error pattern to the bit position it corrupts.

    For each bit index b in 0..codeword_length-1, the syndrome produced by a
    single flip at b is the tuple of parities that include b.  Storing the
    inverse gives an O(1) lookup during decoding.

    Args:
        parity_checks (list of list of int): Parity check index sets.
        codeword_length (int): Number of bits per codeword (7 or 8).

    Returns:
        dict: syndrome tuple -> bit index of the single-bit error.
    """
    syndrome_to_bit = {}
    for bit in range(codeword_length):
        syndrome = tuple(1 if bit in parity_set else 0 for parity_set in parity_checks)
        syndrome_to_bit[syndrome] = bit
    return syndrome_to_bit


def compute_syndrome(codeword, parity_checks):
    """Compute the syndrome of a codeword.

    Each entry of the returned tuple is the XOR of the codeword bits
    indexed by the corresponding parity check set.

    Args:
        codeword (list of str): '0'/'1' characters.
        parity_checks (list of list of int): Parity check index sets.

    Returns:
        tuple of int: Syndrome bits (0 or 1 per check).
    """
    return tuple(
        sum(int(codeword[b]) for b in parity_set) % 2
        for parity_set in parity_checks
    )


def correct_codeword(codeword, parity_checks, syndrome_table, no_of_nodes_in_supernode):
    """Detect and (if possible) correct a single-bit error in one codeword.

    Hamming(7,4):  corrects all single-bit errors.
    Hamming(8,4):  uses the overall parity bit (last check) to distinguish
                   single-bit errors (correctable) from double-bit errors
                   (detectable only).

    Args:
        codeword (list of str): '0'/'1' characters of length no_of_nodes_in_supernode.
        parity_checks (list of list of int): Parity check index sets.
        syndrome_table (dict): Output of build_syndrome_table().
        no_of_nodes_in_supernode (int): 7 or 8.

    Returns:
        tuple:
            - corrected (list of str): Codeword after correction (unchanged if no error).
            - error_bit (int or None): Index of the flipped bit, or None.
            - status (str): Human-readable description.
    """
    syndrome = compute_syndrome(codeword, parity_checks)
    corrected = list(codeword)

    if all(s == 0 for s in syndrome):
        return corrected, None, "no error"

    is_extended = (no_of_nodes_in_supernode == 8)

    if is_extended:
        inner_syndrome = syndrome[:-1]
        overall_parity  = syndrome[-1]

        if all(s == 0 for s in inner_syndrome) and overall_parity == 1:
            # Error is in the overall parity bit itself
            error_bit = no_of_nodes_in_supernode - 1
            corrected[error_bit] = '0' if corrected[error_bit] == '1' else '1'
            return corrected, error_bit, "single-bit error corrected (overall parity bit)"

        if any(s != 0 for s in inner_syndrome) and overall_parity == 1:
            # Single-bit error: locate and correct
            if syndrome in syndrome_table:
                error_bit = syndrome_table[syndrome]
                corrected[error_bit] = '0' if corrected[error_bit] == '1' else '1'
                return corrected, error_bit, f"single-bit error corrected (bit {error_bit})"

        if any(s != 0 for s in inner_syndrome) and overall_parity == 0:
            return corrected, None, "double-bit error detected (uncorrectable)"

    else:
        # Hamming(7,4): every non-zero syndrome maps to exactly one bit
        if syndrome in syndrome_table:
            error_bit = syndrome_table[syndrome]
            corrected[error_bit] = '0' if corrected[error_bit] == '1' else '1'
            return corrected, error_bit, f"single-bit error corrected (bit {error_bit})"

    return corrected, None, f"unknown syndrome {syndrome}"


def correct_hamming_errors(final_answer, N, no_of_nodes_in_supernode, parity_checks):
    """Detect and correct Hamming code errors across the full measurement bitstring.

    Each node occupies a contiguous block of no_of_nodes_in_supernode bits in
    final_answer.  The function applies syndrome decoding independently to
    every node's codeword and returns the fully corrected bitstring.

    Args:
        final_answer (list of str): '0'/'1' chars, length N * no_of_nodes_in_supernode.
        N (int): Number of graph nodes.
        no_of_nodes_in_supernode (int): Bits per node (7 for Hamming(7,4), 8 for Hamming(8,4)).
        parity_checks (list of list of int): Parity check index sets (PARITY_CODES_74 or _84).

    Returns:
        list of str: Corrected bitstring of the same length as final_answer.
    """
    syndrome_table = build_syndrome_table(parity_checks, no_of_nodes_in_supernode)
    corrected_answer = list(final_answer)

    print(f"\nHamming({no_of_nodes_in_supernode},4) Syndrome Decoding")
    print(f"{'Node':<6} {'Codeword':<{no_of_nodes_in_supernode+2}} {'Syndrome':<{len(parity_checks)+2}} {'Status':<42} Corrected")
    print("-" * (no_of_nodes_in_supernode * 2 + len(parity_checks) + 56))

    errors_found = 0
    for i in range(N):
        start = i * no_of_nodes_in_supernode
        codeword = final_answer[start : start + no_of_nodes_in_supernode]
        syndrome = compute_syndrome(codeword, parity_checks)
        corrected, error_bit, status = correct_codeword(
            codeword, parity_checks, syndrome_table, no_of_nodes_in_supernode
        )
        corrected_answer[start : start + no_of_nodes_in_supernode] = corrected

        changed = "  *" if corrected != list(codeword) else ""
        if error_bit is not None:
            errors_found += 1
        print(
            f"{i:<6} {''.join(codeword):<{no_of_nodes_in_supernode+2}} "
            f"{''.join(map(str, syndrome)):<{len(parity_checks)+2}} "
            f"{status:<42} {''.join(corrected)}{changed}"
        )

    print()
    if errors_found == 0:
        print("All codewords valid — no corrections needed.")
    else:
        print(f"{errors_found} node(s) corrected.")
        print(f"Corrected bitstring: {''.join(corrected_answer)}")

    return corrected_answer

**QAOA implementation**

Hamming(7,4)

In [ ]:
def run_Hamming_74(G, N, K, LAMBDA_PENALTY, PARITY_CODES_74):
    """Runs the full QAOA pipeline for community detection using Hamming(7,4) encoding.

    Constructs the supernode graph, modularity and parity penalty Hamiltonians,
    and the QAOA ansatz. Runs classical optimisation on a local AerSimulator,
    then samples the optimised circuit and decodes the result using Hamming(7,4)
    syndrome correction to obtain final community assignments.

    Args:
        G (networkx.Graph): The problem graph.
        N (int): The number of nodes in the graph.
        K (int): The number of communities.
        LAMBDA_PENALTY (float): Penalty coefficient for the parity constraint Hamiltonian.
        PARITY_CODES_74 (list): Parity check sets defining the Hamming(7,4) code structure.

    Returns:
        list: The final community assignment bitstring decoded from the optimised circuit.
    """
    display_graph(G, title="Initial Graph for Community Detection") # Visualise the problem graph

    no_of_nodes_in_supernode = 7 # Hamming(7,4) uses 7-bit codewords, so each node maps to 7 qubits

    G_onehot_supernodes_labeled = create_supernode(G, no_of_nodes_in_supernode) # Create the supernode graph with 7 copies per original node
    display_graph(G_onehot_supernodes_labeled, "Supernode Structure") # Visualise the supernode structure

    modularity_matrix = create_modularity_matrix(G, N) # Calculate the modularity matrix for the graph

    num_qubits, modularity_hamiltonian = create_modularity_penalty_hamiltonian(G, N, modularity_matrix, K, no_of_nodes_in_supernode) # Construct the modularity Hamiltonian encoding the community objective

    _, penalty_hamiltonian = create_parity_penalty_hamiltonian(N, no_of_nodes_in_supernode, PARITY_CODES_74, LAMBDA_PENALTY) # Construct the parity penalty Hamiltonian enforcing valid Hamming(7,4) codewords

    total_hamiltonian = create_total_cost_hamiltonian(modularity_hamiltonian, penalty_hamiltonian, num_qubits) # Combine modularity and parity terms into the total objective Hamiltonian

    # QAOA ansatz construction
    circuit = create_qaoa_ansatz(total_hamiltonian, P, N, K) # Create the QAOA quantum circuit ansatz
    print(f"\n\nQAOA Ansatz: ")
    draw_circuit_mpl(circuit, "QAOA Ansatz Circuit") # Draw and display the QAOA ansatz circuit

    # Optimisation process
    initial_params = [INITIAL_GAMMA] * P + [INITIAL_BETA] * P # Set initial parameters for the QAOA optimisation
    transpiled_qc, backend = run_ansatz_simulator_circuit_prep(circuit) # Prepare the circuit for simulation (transpile for AerSimulator)
    result, value_tracker = run_optimizer_simulator(transpiled_qc, backend, initial_params, total_hamiltonian) # Run the classical optimiser to find optimal QAOA parameters
    plot_optimizer_results(value_tracker) # Plot the cost function's value over optimisation iterations

    # Result extraction and interpretation
    optimized_circuit = transpiled_qc.assign_parameters(result.x) # Assign the optimised parameters to the quantum circuit
    print(f"\n\nThe circuit with optimized parameters: ")
    draw_circuit_mpl(optimized_circuit, "Optimized Circuit") # Draw and display the optimised circuit

    # Sample measurement outcomes, apply syndrome correction, and decode community assignments from data bits
    final_answer = sample_output(G, optimized_circuit, backend, num_qubits,
                                 no_of_nodes_in_supernode, PARITY_CODES_74)
    return final_answer


G, N = create_clique_chain_graph(3,3)
K = 4
LAMBDA_PENALTY = 5 # Penalty parameter for the Hamiltonian to enforce constraints
final_answer = run_Hamming_74(G, N, K, LAMBDA_PENALTY, PARITY_CODES_74)

In [ ]:
#Rough work

PARITY_CHECK_SETS = [
    (0, [2, 4, 6]),   # parity bit at pos 0 checks positions 2, 4, 6
    (1, [2, 5, 6]),   # parity bit at pos 1 checks positions 2, 5, 6
    (3, [4, 5, 6]),   # parity bit at pos 3 checks positions 4, 5, 6
]

def count_parity_violations(bitstring: str, n_nodes: int) -> int:
    """Count how many parity checks are violated in the bitstring."""
    bits = list(reversed(bitstring))
    violations = 0 
    
    for node in range(n_nodes):
        for parity_pos, checked_positions in PARITY_CHECK_SETS:
            p_bit = int(bits[node * 7 + parity_pos])
            data_xor = 0
            for dp in checked_positions:
                data_xor ^= int(bits[node * 7 + dp])
            if p_bit != data_xor:
                violations += 1
    
    return violations

count_parity_violations("".join(final_answer), 6)

Classical Simulation

$H_{mod} = +Σ_q Σ_{i,j} B_{ij} * (2x_{i,q}-1)(2x_{j,q}-1)$

In [ ]:
'''
Classical simulation — Hamming(7,4)
Total objective to MINIMISE:

  H_total = H_mod + PP

  H_mod = +Σ_q Σ_{i,j} B_ij * (2x_{i,q}-1)(2x_{j,q}-1)   (modularity)
  PP    = -λ * Σ_i Σ_{p∈P} Π_{q∈p} (2x_{i,q}-1)           (parity penalty)

x_{i,q} ∈ {0, 1}  — strictly binary, no relaxation
v_{i,q} = 2x_{i,q} - 1  maps {0,1} → {-1,+1}

Optimiser: simulated annealing over {0,1}^(N×7) with random bit-flip moves.
           Operates directly on binary strings — no rounding needed at decode.
'''

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

NO_OF_BITS = 7
DATA_BITS = [2, 4, 5, 6]


def build_modularity_matrix_hamming(G):
    """Constructs the modularity matrix B for the Hamming-encoded community detection problem.

    Computes B = A - (deg * deg^T) / (2m), where A is the adjacency matrix,
    deg is the degree vector, and m is the total number of edges.

    Args:
        G (networkx.Graph): The original graph for which the modularity matrix is derived.

    Returns:
        tuple: A tuple containing:
            - B (numpy.ndarray): The NxN modularity matrix.
            - m (float): The total number of edges in the graph.
    """
    nodes = sorted(G.nodes())
    A = nx.to_numpy_array(G, nodelist=nodes) # Convert graph to adjacency matrix with consistent node ordering
    m = A.sum() / 2 # Total number of edges (sum of all entries divided by 2)
    deg = A.sum(axis=1) # Degree vector: sum of each row gives the degree of each node
    B = A - np.outer(deg, deg) / (2 * m) # Modularity matrix: B_ij = A_ij - k_i*k_j / (2m)
    return B, m


# Objective on binary {0,1} inputs
def hamming_objective(x_flat, B, n, lam):
    """Computes the Hamming-encoded total objective for simulated annealing.

    Evaluates H_total = -H_mod + PP, where H_mod is the modularity gain and
    PP is the parity penalty. The modularity term is negated so that
    minimising H_total corresponds to maximising modularity subject to
    valid Hamming codewords.

    Args:
        x_flat (numpy.ndarray): Flattened binary assignment matrix of shape (n*NO_OF_BITS,).
        B (numpy.ndarray): The modularity matrix for the graph.
        n (int): The number of nodes in the graph.
        lam (float): Penalty coefficient controlling the strength of the parity constraint.

    Returns:
        float: The total objective value H_total (to be minimised).
    """
    X = x_flat.reshape(n, NO_OF_BITS)
    V = 2.0 * X - 1.0 # Map {0,1} → {-1,+1} as required by the modularity quadratic form

    h_mod = sum(V[:, q] @ B @ V[:, q] for q in range(NO_OF_BITS)) # Sum modularity contributions across all bit positions
    pp = -lam * sum(np.prod(V[i, pc]) for i in range(n) for pc in PARITY_CODES_74) # Parity penalty: penalise codewords violating Hamming(7,4) parity checks

    return -h_mod + pp # Negate h_mod to convert maximisation to minimisation


# Simulated annealing over {0,1}^(N×7)
def run_binary_sa(B, n, lam, T_init=5.0, T_min=1e-4, maxiter=100000, seed=42):
    """Minimises the Hamming objective using simulated annealing over binary strings.

    At each step a single random bit is flipped and the move is accepted via
    the Metropolis criterion. Temperature decreases geometrically from T_init
    to T_min over maxiter steps.

    Args:
        B (numpy.ndarray): The modularity matrix for the graph.
        n (int): The number of nodes in the graph.
        lam (float): Penalty coefficient for the parity constraint.
        T_init (float): Initial temperature for the annealing schedule.
        T_min (float): Final temperature at which annealing stops.
        maxiter (int): Total number of bit-flip proposals.
        seed (int): Random seed for reproducibility.

    Returns:
        tuple: A tuple containing:
            - best_x (numpy.ndarray): The best binary solution found.
            - best_val (float): The lowest objective value achieved.
            - tracker (list[float]): Best-so-far objective value at each improvement.
    """
    rng = np.random.default_rng(seed)
    n_vars = n * NO_OF_BITS
    cooling = (T_min / T_init) ** (1.0 / maxiter) # Geometric cooling factor applied each iteration

    x = rng.integers(0, 2, size=n_vars).astype(float) # Random binary initialisation
    val = hamming_objective(x, B, n, lam)
    best_x = x.copy()
    best_val = val
    tracker = [val] # Best-so-far objective value, updated at each improvement

    T = T_init
    for _ in range(maxiter):
        # Propose: flip one random bit
        idx = rng.integers(0, n_vars)
        x_new = x.copy()
        x_new[idx] = 1.0 - x_new[idx]

        new_val = hamming_objective(x_new, B, n, lam)
        delta = new_val - val

        # Metropolis acceptance: always accept improvements, accept worse moves with probability exp(-delta/T)
        if delta < 0 or rng.random() < np.exp(-delta / T):
            x = x_new
            val = new_val
            if val < best_val:
                best_val = val
                best_x = x.copy()
                tracker.append(best_val) # Record new best value

        T *= cooling # Cool the temperature

    print(f"SA finished | best H_total = {best_val:.4f} | improvements = {len(tracker)-1}")
    return best_x, best_val, tracker


# Syndrome correction
def syndrome_correct_74(codeword_bits):
    """Performs single-bit error correction using the Hamming(7,4) syndrome.

    Computes a 3-bit syndrome by XORing the bits covered by each parity
    check. The syndrome value directly encodes the position of the erroneous
    bit (1-indexed), which is then flipped to correct the error.

    Args:
        codeword_bits (list[int]): A list of 7 binary integers representing the received codeword.

    Returns:
        list[int]: The corrected codeword as a list of 7 binary integers.
    """
    bits = list(codeword_bits)
    syndrome = [
        int(np.bitwise_xor.reduce([bits[q] for q in pc])) % 2
        for pc in PARITY_CODES_74
    ]
    error_pos = syndrome[2] * 4 + syndrome[1] * 2 + syndrome[0] - 1 # Decode syndrome to 0-indexed error position (-1 means no error)
    if error_pos >= 0:
        bits[error_pos] ^= 1 # Flip the erroneous bit to correct it
    return bits


# Decode binary solution → community assignments
def decode_hamming_assignments(x_flat, n, k):
    """Decodes the binary SA solution into integer community assignments using Hamming(7,4).

    For each node, applies syndrome correction to its 7-bit codeword, extracts
    the 4 data bits, interprets them as a binary integer, and maps the result
    into the valid community range [0, k) via modulo.

    Args:
        x_flat (numpy.ndarray): Flattened binary solution of shape (n*NO_OF_BITS,).
        n (int): The number of nodes in the graph.
        k (int): The number of communities.

    Returns:
        numpy.ndarray: Integer array of shape (n,) giving the community index for each node.
    """
    X = x_flat.reshape(n, NO_OF_BITS).astype(int)

    assignments = []
    print("Syndrome Correction Statistics ")
    print(f"\n{'Node':<6} {'Codeword':<9} {'After correction':<17} {'Data bits':<11} Community")
    print("-" * 52)
    
    for i in range(n):
        raw = list(X[i])
        corrected = syndrome_correct_74(raw) # Apply Hamming(7,4) syndrome correction
        data_bits = [corrected[q] for q in DATA_BITS] # Extract the 4 data bits from the corrected codeword
        value = int("".join(str(b) for b in data_bits), 2) # Interpret data bits as a big-endian binary integer
        community = value % k # Wrap integer into valid community range [0, k)
        assignments.append(community)
        print(f"{i:<6} {''.join(map(str, raw)):<9} {''.join(map(str, corrected)):<17} "
              f"{''.join(map(str, data_bits)):<11} {community}")

    return np.array(assignments)


# Visualisation
def plot_hamming_convergence(tracker):
    """Plots the best-so-far objective value over the course of simulated annealing.

    Args:
        tracker (list[float]): Sequence of best objective values recorded at each improvement.
    """
    plt.figure(figsize=(10, 4))
    plt.step(range(len(tracker)), tracker, where="post", color="red", linewidth=1.5)
    plt.xlabel("Improvement number")
    plt.ylabel("Best H_total found")
    plt.title("Simulated annealing — Hamming(7,4) best-so-far")
    plt.tight_layout()
    plt.show()


def plot_hamming_communities(G, assignments, k):
    """Visualises the detected community structure of the graph using colour-coded nodes.

    Args:
        G (networkx.Graph): The original problem graph.
        assignments (numpy.ndarray): Integer array of community indices for each node.
        k (int): The number of communities.
    """
    cmap = plt.get_cmap("tab10")
    node_colors = [cmap(c % cmap.N) for c in assignments] # Assign a colour to each node based on its community
    pos = nx.spring_layout(G, seed=42)
    plt.figure(figsize=(8, 6))
    nx.draw_networkx_edges(G, pos, edge_color="gray", width=2)
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=700)
    nx.draw_networkx_labels(G, pos, font_color="white", font_size=10)
    plt.title(f"Detected Communities — Hamming(7,4), K={k}")
    plt.axis("off")
    plt.show()


def run_hamming74_classical(G, N, K, LAMBDA_PENALTY):
    """Runs the full Hamming(7,4) classical simulated annealing pipeline.

    Constructs the modularity matrix, runs simulated annealing to find the
    optimal binary assignment, decodes the result using Hamming(7,4) syndrome
    correction, and displays the convergence plot and final community partition.

    Args:
        G (networkx.Graph): The problem graph.
        N (int): The number of nodes in the graph.
        K (int): The number of communities.
        LAMBDA_PENALTY (float): Penalty coefficient for the parity constraint.
    """
    display_graph(G) # Visualise the problem graph before optimisation
    B_ham, m_ham = build_modularity_matrix_hamming(G) # Construct the modularity matrix and retrieve the edge count
    print(f"\nModularity matrix B = {B_ham}")

    best_x, best_val, tracker = run_binary_sa(B_ham, N, LAMBDA_PENALTY) # Run simulated annealing to find the optimal binary assignment
    assignments = decode_hamming_assignments(best_x, N, K) # Decode binary solution into community assignments using syndrome correction

    print(f"\nOptimal H_total     : {best_val:.4f}")
    print(f"Community assignments: {assignments}")

    plot_hamming_convergence(tracker) # Plot the best-so-far objective over annealing improvements
    plot_hamming_communities(G, assignments, K) # Visualise the final community partition on the graph


G, N = get_davis_southern_women_graph()
K = 2
LAMBDA_PENALTY = 5 # Penalty parameter for the Hamiltonian to enforce constraints

run_hamming74_classical(G, N, K, LAMBDA_PENALTY)

Hamming(8,4)

In [ ]:
def run_Hamming_84(G, N, K, LAMBDA_PENALTY):
    """Runs the full QAOA pipeline for community detection using Hamming(8,4) encoding.

    Constructs the supernode graph, modularity and parity penalty Hamiltonians,
    and the QAOA ansatz. Runs classical optimisation on a local AerSimulator,
    then samples the optimised circuit and decodes the result using Hamming(8,4)
    syndrome correction to obtain final community assignments.

    Args:
        G (networkx.Graph): The problem graph.
        N (int): The number of nodes in the graph.
        K (int): The number of communities.
        LAMBDA_PENALTY (float): Penalty coefficient for the parity constraint Hamiltonian.

    Returns:
        list: The final community assignment bitstring decoded from the optimised circuit.
    """
    display_graph(G, title="Initial Graph for Community Detection") # Visualise the problem graph

    no_of_nodes_in_supernode = 8 # Hamming(8,4) uses 8-bit codewords, so each node maps to 8 qubits

    G_onehot_supernodes_labeled = create_supernode(G, no_of_nodes_in_supernode) # Create the supernode graph with 8 copies per original node
    display_graph(G_onehot_supernodes_labeled, "Supernode Structure") # Visualise the supernode structure

    modularity_matrix = create_modularity_matrix(G, N) # Calculate the modularity matrix for the graph

    num_qubits, modularity_hamiltonian = create_modularity_penalty_hamiltonian(G, N, modularity_matrix, K, no_of_nodes_in_supernode) # Construct the modularity Hamiltonian encoding the community objective

    _, penalty_hamiltonian = create_parity_penalty_hamiltonian(N, no_of_nodes_in_supernode, PARITY_CODES_84, LAMBDA_PENALTY) # Construct the parity penalty Hamiltonian enforcing valid Hamming(8,4) codewords

    total_hamiltonian = create_total_cost_hamiltonian(modularity_hamiltonian, penalty_hamiltonian, num_qubits) # Combine modularity and parity terms into the total objective Hamiltonian

    # QAOA ansatz construction
    circuit = create_qaoa_ansatz(total_hamiltonian, P, N, K) # Create the QAOA quantum circuit ansatz
    print(f"\n\nQAOA Ansatz: ")
    draw_circuit_mpl(circuit, "QAOA Ansatz Circuit") # Draw and display the QAOA ansatz circuit

    # Optimisation process
    initial_params = [INITIAL_GAMMA] * P + [INITIAL_BETA] * P # Set initial parameters for the QAOA optimisation
    transpiled_qc, backend = run_ansatz_simulator_circuit_prep(circuit, "mps") # Prepare the circuit for MPS simulation (efficient for large qubit counts)
    result, value_tracker = run_optimizer_simulator(transpiled_qc, backend, initial_params, total_hamiltonian) # Run the classical optimiser to find optimal QAOA parameters
    plot_optimizer_results(value_tracker) # Plot the cost function's value over optimisation iterations

    # Result extraction and interpretation
    optimized_circuit = transpiled_qc.assign_parameters(result.x) # Assign the optimised parameters to the quantum circuit
    print(f"\n\nThe circuit with optimized parameters: ")
    draw_circuit_mpl(optimized_circuit, "Optimized Circuit") # Draw and display the optimised circuit

    # Sample measurement outcomes, apply syndrome correction, and decode community assignments from data bits
    final_answer = sample_output(G, optimized_circuit, backend, num_qubits,
                                 no_of_nodes_in_supernode, PARITY_CODES_84)
    return final_answer


G, N = create_clique_chain_graph(3, 3)
K = 2
LAMBDA_PENALTY = 5 # Penalty parameter for the Hamiltonian to enforce constraints

run_Hamming_84(G, N, K, LAMBDA_PENALTY)

Classical Simulation

In [ ]:
'''

Classical simulation — Hamming(8,4)
Total objective to MINIMISE:

  H_total = -H_mod(reason being it's converted into quadratic form) + PP

  H_mod = +Σ_q Σ_{i,j} B_ij * (2x_{i,q}-1)(2x_{j,q}-1)   (modularity)
  PP    = -λ * Σ_i Σ_{p∈P} Π_{q∈p} (2x_{i,q}-1)           (parity penalty)

x_{i,q} ∈ {0, 1}  — strictly binary, no relaxation
v_{i,q} = 2x_{i,q} - 1  maps {0,1} → {-1,+1}

Optimiser: simulated annealing over {0,1}^(N×8) with random bit-flip moves.
           Operates directly on binary strings — no rounding needed at decode.
'''
           
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

NO_OF_BITS = 8
DATA_BITS = [2, 4, 5, 6]


def build_modularity_matrix_hamming(G):
    """Constructs the modularity matrix B for the Hamming-encoded community detection problem.

    Computes B = A - (deg * deg^T) / (2m), where A is the adjacency matrix,
    deg is the degree vector, and m is the total number of edges.

    Args:
        G (networkx.Graph): The original graph for which the modularity matrix is derived.

    Returns:
        tuple: A tuple containing:
            - B (numpy.ndarray): The NxN modularity matrix.
            - m (float): The total number of edges in the graph.
    """
    nodes = sorted(G.nodes())
    A = nx.to_numpy_array(G, nodelist=nodes) # Convert graph to adjacency matrix with consistent node ordering
    m = A.sum() / 2 # Total number of edges (sum of all entries divided by 2)
    deg = A.sum(axis=1) # Degree vector: sum of each row gives the degree of each node
    B = A - np.outer(deg, deg) / (2 * m) # Modularity matrix: B_ij = A_ij - k_i*k_j / (2m)
    return B, m


# Objective on binary {0,1} inputs 
def hamming_objective(x_flat, B, n, lam):
    """Computes the Hamming(8,4)-encoded total objective for simulated annealing.

    Evaluates H_total = -H_mod + PP, where H_mod is the modularity gain and
    PP is the parity penalty. The modularity term is negated so that
    minimising H_total corresponds to maximising modularity subject to
    valid Hamming(8,4) codewords.

    Args:
        x_flat (numpy.ndarray): Flattened binary assignment matrix of shape (n*NO_OF_BITS,).
        B (numpy.ndarray): The modularity matrix for the graph.
        n (int): The number of nodes in the graph.
        lam (float): Penalty coefficient controlling the strength of the parity constraint.

    Returns:
        float: The total objective value H_total (to be minimised).
    """
    X = x_flat.reshape(n, NO_OF_BITS)
    V = 2.0 * X - 1.0 # Map {0,1} → {-1,+1} as required by the modularity quadratic form

    h_mod = sum(V[:, q] @ B @ V[:, q] for q in range(NO_OF_BITS)) # Sum modularity contributions across all bit positions
    pp = -lam * sum(np.prod(V[i, pc]) for i in range(n) for pc in PARITY_CODES_84) # Parity penalty: penalise codewords violating Hamming(8,4) parity checks

    return -h_mod + pp # Negate h_mod to convert maximisation to minimisation


# Simulated annealing over {0,1}^(N×8) 
def run_binary_sa(B, n, lam, T_init=5.0, T_min=1e-4, maxiter=100000, seed=42):
    """Minimises the Hamming(8,4) objective using simulated annealing over binary strings.

    At each step a single random bit is flipped and the move is accepted via
    the Metropolis criterion. Temperature decreases geometrically from T_init
    to T_min over maxiter steps.

    Args:
        B (numpy.ndarray): The modularity matrix for the graph.
        n (int): The number of nodes in the graph.
        lam (float): Penalty coefficient for the parity constraint.
        T_init (float): Initial temperature for the annealing schedule.
        T_min (float): Final temperature at which annealing stops.
        maxiter (int): Total number of bit-flip proposals.
        seed (int): Random seed for reproducibility.

    Returns:
        tuple: A tuple containing:
            - best_x (numpy.ndarray): The best binary solution found.
            - best_val (float): The lowest objective value achieved.
            - tracker (list[float]): Best-so-far objective value at each improvement.
    """
    rng = np.random.default_rng(seed)
    n_vars = n * NO_OF_BITS
    cooling = (T_min / T_init) ** (1.0 / maxiter) # Geometric cooling factor applied each iteration

    x = rng.integers(0, 2, size=n_vars).astype(float) # Random binary initialisation
    val = hamming_objective(x, B, n, lam)
    best_x = x.copy()
    best_val = val
    tracker = [val] # Best-so-far objective value, updated at each improvement

    T = T_init
    for _ in range(maxiter):
        # Propose: flip one random bit
        idx = rng.integers(0, n_vars)
        x_new = x.copy()
        x_new[idx] = 1.0 - x_new[idx]

        new_val = hamming_objective(x_new, B, n, lam)
        delta = new_val - val

        # Metropolis acceptance: always accept improvements, accept worse moves with probability exp(-delta/T)
        if delta < 0 or rng.random() < np.exp(-delta / T):
            x = x_new
            val = new_val
            if val < best_val:
                best_val = val
                best_x = x.copy()
                tracker.append(best_val) # Record new best value

        T *= cooling # Cool the temperature

    print(f"SA finished | best H_total = {best_val:.4f} | improvements = {len(tracker)-1}")
    return best_x, best_val, tracker


# Syndrome correction
def syndrome_correct_84(codeword_bits):
    """Performs single-bit error correction using the Hamming(8,4) syndrome.

    Computes a 3-bit syndrome from parity bits P1, P2, P3 to locate a single-bit
    error. The overall parity bit P4 (XOR of all 8 bits) then distinguishes
    single-bit errors (correctable) from double-bit errors (detectable only).

    Args:
        codeword_bits (list[int]): A list of 8 binary integers representing the received codeword.

    Returns:
        list[int]: The corrected codeword as a list of 8 binary integers.
    """
    bits = list(codeword_bits)

    # 3-bit syndrome from P1, P2, P3 only (indices 0,1,2 of PARITY_CODES_84)
    syndrome = [
        int(np.bitwise_xor.reduce([bits[q] for q in PARITY_CODES_84[k]])) % 2
        for k in range(3)
    ]
    # P4: XOR of all 8 bits — used to distinguish single-bit from double-bit errors
    p4 = int(np.bitwise_xor.reduce(bits)) % 2

    error_pos = syndrome[2] * 4 + syndrome[1] * 2 + syndrome[0] - 1 # Decode syndrome to 0-indexed error position (-1 means no error)

    if p4 == 1 and error_pos >= 0:      # Single-bit error — correct it
        bits[error_pos] ^= 1
    elif p4 == 0 and error_pos >= 0:    # Double-bit error — detect only, cannot correct
        print("  Warning: double-bit error detected, cannot correct.")
    # p4==0 & error_pos<0 → no error

    return bits


# Decode binary solution → community assignments 
def decode_hamming_assignments_84(x_flat, n, k):
    """Decodes the binary SA solution into integer community assignments using Hamming(8,4).

    For each node, applies Hamming(8,4) syndrome correction to its 8-bit codeword,
    extracts the 4 data bits, interprets them as a binary integer, and maps the
    result into the valid community range [0, k) via modulo.

    Args:
        x_flat (numpy.ndarray): Flattened binary solution of shape (n*NO_OF_BITS,).
        n (int): The number of nodes in the graph.
        k (int): The number of communities.

    Returns:
        numpy.ndarray: Integer array of shape (n,) giving the community index for each node.
    """
    X = x_flat.reshape(n, NO_OF_BITS).astype(int)

    assignments = []
    print("Syndrome Correction Statistics ")
    print(f"\n{'Node':<6} {'Codeword':<9} {'After correction':<17} {'Data bits':<11} Community")
    print("-" * 52)
    for i in range(n):
        raw = list(X[i])
        corrected = syndrome_correct_84(raw) # Apply Hamming(8,4) syndrome correction
        data_bits = [corrected[q] for q in DATA_BITS] # Extract the 4 data bits from the corrected codeword
        value = int("".join(str(b) for b in data_bits), 2) # Interpret data bits as a big-endian binary integer
        community = value % k # Wrap integer into valid community range [0, k)
        assignments.append(community)
        print(f"{i:<6} {''.join(map(str, raw)):<9} {''.join(map(str, corrected)):<17} "
              f"{''.join(map(str, data_bits)):<11} {community}")

    return np.array(assignments)


# Visualisation 
def plot_hamming_convergence(tracker):
    """Plots the best-so-far objective value over the course of simulated annealing.

    Args:
        tracker (list[float]): Sequence of best objective values recorded at each improvement.
    """
    plt.figure(figsize=(10, 4))
    plt.step(range(len(tracker)), tracker, where="post", color="red", linewidth=1.5)
    plt.xlabel("Improvement number")
    plt.ylabel("Best H_total found")
    plt.title("Simulated annealing — Hamming(8,4) best-so-far")
    plt.tight_layout()
    plt.show()


def plot_hamming_communities(G, assignments, k):
    """Visualises the detected community structure of the graph using colour-coded nodes.

    Args:
        G (networkx.Graph): The original problem graph.
        assignments (numpy.ndarray): Integer array of community indices for each node.
        k (int): The number of communities.
    """
    cmap = plt.get_cmap("tab10")
    node_colors = [cmap(c % cmap.N) for c in assignments] # Assign a colour to each node based on its community
    pos = nx.spring_layout(G, seed=42)
    plt.figure(figsize=(8, 6))
    nx.draw_networkx_edges(G, pos, edge_color="gray", width=2)
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=700)
    nx.draw_networkx_labels(G, pos, font_color="white", font_size=10)
    plt.title(f"Detected Communities — Hamming(8,4), K={k}")
    plt.axis("off")
    plt.show()

def run_hamming84_classical(G, N, K, LAMBDA_PENALTY ):
    """Runs the full Hamming(8,4) classical simulated annealing pipeline.

    Constructs the modularity matrix, runs simulated annealing to find the
    optimal binary assignment, decodes the result using Hamming(7,4) syndrome
    correction, and displays the convergence plot and final community partition.

    Args:
        G (networkx.Graph): The problem graph.
        N (int): The number of nodes in the graph.
        K (int): The number of communities.
        LAMBDA_PENALTY (float): Penalty coefficient for the parity constraint.
    """
    display_graph(G) # Visualise the problem graph before optimisation
    B, m = build_modularity_matrix_hamming(G) # Construct the modularity matrix and retrieve the edge count
    print(f"\nModularity matrix B = {B}")

    best_x, best_val, tracker = run_binary_sa(B, N, LAMBDA_PENALTY) # Run simulated annealing to find the optimal binary assignment
    assignments = decode_hamming_assignments_84(best_x, N, K) # Decode binary solution into community assignments using syndrome correction

    print(f"\nOptimal H_total     : {best_val:.4f}")
    print(f"Community assignments: {assignments}")

    plot_hamming_convergence(tracker) # Plot the best-so-far objective over annealing improvements
    plot_hamming_communities(G, assignments, K) # Visualise the final community partition on the graph

G, N = create_clique_chain_graph(3, 2)
K = 2
LAMBDA_PENALTY = 1 # Penalty parameter for the Hamiltonian to enforce constraints

run_hamming84_classical(G, N, K, LAMBDA_PENALTY )